En primer lugar, vamos a realizar las importaciones necesarias para poder empezar a trabajar: 

In [2]:
#Importamos pandas
import pandas as pd 
#Importamos numpy
import numpy as np 
#Importamos nuestro archivo de soporte
import sys
sys.path.append('../')
from Jupyters.soporte_def import *

print ("Librerias importadas")

Librerias importadas


Una vez importadas las librerías de python necesarias, procedemos a importar el dataset original con el que trabajaremos y a hacer una copia del mismo para no modificar los datos reales.

In [3]:
#En primer lugar importamos el dataset
df_original = pd.read_csv ("../Clean_Data/Dataframe_sin_nulos.csv")
print ("Dataframe importado")

Dataframe importado


A continuación, procedemos a copiar el dataframe a uno de trabajo.

In [4]:
df_work = df_original
print ("Dataframe copiado")

Dataframe copiado


Durante esta segunda fase de la limpieza, vamos a proceder a analizar los duplicados tras eliminar los nulos.

***Paso 3: Registro y eliminación de duplicados***

Lo primero que vamos a obtener es como en el caso anterior una tabla que nos indique el número de duplicados y porcentaje de duplicados por columna:

In [5]:
analisis_duplicados(df_work)

-----DUPLICADOS POR COLUMNA-----
              columna  duplicados  porcentaje
0                  id       86258       98.15
1        general_team       87849       99.96
2        general_city       87844       99.96
3   general_team_name       87844       99.96
4   general_team_abbv       87840       99.95
5           full_name       86261       98.16
6           is_active       87879      100.00
7          draft_team       87850       99.96
8          draft_city       87846       99.96
9     draft_team_name       87847       99.96
10   draft_team_abbrv       87841       99.95
11  organization_type       87877      100.00
12          birthdate       86347       98.25
13            country       87816       99.93
14             weight       87758       99.86
15          seniority       87858       99.97
16           position       87874       99.99
17          from_year       87841       99.95
18            to_year       87853       99.97
19           g-league       87879      100.00
2

Según parece hay bastantes registros duplicados por lo que la primera limpieza que vamos a hacer va a ser la de jugadores y equipos duplicados. 

De esta manera, nos quedaremos con una sola línea por jugador y equipo. Es decir solo nos quedará una fila por cada equipo en el que haya jugado cada jugador.

In [6]:
#Obtenemos la forma del dataframe antes de la eliminación
df_work.shape

#Para hacer esta limpieza nos vamos a apoyar de la función drop_duplicates
df_work = df_work.drop_duplicates(subset= ['id', 'general_team']).reset_index(drop = True)

#Obtenemos la forma del dataframe después 
df_work.shape

(3712, 28)

Una vez borrado, volvemos a analizar los duplicados columna a columna para ver como gestionarlos:

In [7]:
analisis_duplicados(df_work)

-----DUPLICADOS POR COLUMNA-----
              columna  duplicados  porcentaje
0                  id        2089       56.28
1        general_team        3680       99.14
2        general_city        3675       99.00
3   general_team_name        3676       99.03
4   general_team_abbv        3672       98.92
5           full_name        2092       56.36
6           is_active        3710       99.95
7          draft_team        3681       99.16
8          draft_city        3677       99.06
9     draft_team_name        3678       99.08
10   draft_team_abbrv        3672       98.92
11  organization_type        3708       99.89
12          birthdate        2178       58.67
13            country        3647       98.25
14             weight        3589       96.69
15          seniority        3689       99.38
16           position        3705       99.81
17          from_year        3672       98.92
18            to_year        3684       99.25
19           g-league        3710       99.95
2

Tras el estudio, parece un gran número de duplicados en cada columna pero si nos detenemos a entender el origen de este dataframe veremos el por que. 

En realidad, estamos tratando con un dataframe que considera solo 1623 jugadores por lo que es normal encontrar un tan alto número de duplicados. 

Por otro lado, el factor diferencial de cara a realizar el estudio es que no se den filas repetidas que por lo que podemos observar así es.

***Paso 4 --> Data consistency***

En esta parte de la limpieza, vamos a ver que las columnas tengan el tipo correcto asignado y entender los valores únicos en cada una de las columnas categóricas. 

Además, también realizaremos las pequeñas modificaciones necesarias en términos de mayúsculas minúsculas y formatos de ser necesarias.

*1. Revisión de tipos de columnas*

Lo primero que vamos a analizar es el tipo de columnas que tenemos y ver que todas sean correctas:

In [8]:
df_work.dtypes

id                     int64
general_team           int64
general_city             str
general_team_name        str
general_team_abbv        str
full_name                str
is_active            float64
draft_team           float64
draft_city               str
draft_team_name          str
draft_team_abbrv         str
organization_type        str
birthdate                str
country                  str
weight               float64
seniority            float64
position                 str
from_year            float64
to_year              float64
g-league                 str
nba                      str
draft_year               str
draft_round              str
draft_number             str
greatest_75_flag         str
jersey_def           float64
organization_def         str
height_definitivo    float64
dtype: object

El primer error que observamos está en la columna is_active que es una columna categórica guardada como una columna de tipo string. Vamos a ver a que se debe extrayendo los valores unicos de dicha columna:

In [9]:
df_work['is_active'].unique()

array([0., 1.])

Como podemos observar, la columna informa de si un jugador está activo o no a través de números por lo que anotaremos este error para sustituirlo por un yes o no durante la fase de transformación.

El segundo error que observamos es que la columna birthdate está en formato string cuando debería estar en formato datetime. Vamos a ver a que se debe visualizando los primeros registros de la columna.

In [10]:
df_work['birthdate'].head(5)

0    31/03/1969 00:00
1    31/10/1972 00:00
2    15/06/1970 00:00
3    07/02/1970 00:00
4    27/03/1963 00:00
Name: birthdate, dtype: str

El error se debe a que la columna informa de la fecha y también incluye erróneamente una hora. Como en el caso anterior anotaremos el error para corregirlo durante la fase de transformación.

Quitados estos dos errores con respecto al tipo de columnas no observamos ningún error adicional.

*2. Obtención de valores únicos de las columnas categóricas*

En este paso nuestro objetivo es ganar un mayor entendimiento del dataframe entendiendo los valores unicos que forman las columnas categóricas. Para hacerlo vamos a utilizar la función value_counts de manera que también entenderemos cuantas veces se repite cada valor:

- Columna is_active:

In [12]:
df_work['is_active'].value_counts()

is_active
0.0    2690
1.0    1022
Name: count, dtype: int64

-  Columna organization_type

In [13]:
df_work['organization_type'].value_counts()

organization_type
College/University    2396
Unknown                878
Other Team/Club        311
High School            127
Name: count, dtype: int64

- Columnas de equipos

In [15]:
df_work['general_team_abbv'].value_counts()

general_team_abbv
HOU    156
CLE    145
NYK    142
DAL    137
MIL    134
PHI    133
ATL    130
BOS    130
SAC    128
DET    125
WAS    124
CHI    123
TOR    120
MIA    118
LAL    118
LAC    117
GSW    116
MEM    116
SAS    115
MIN    113
PHX    111
IND    108
UTA    107
CHA    107
OKC    104
POR    101
ORL     98
DEN     93
BKN     83
NOP     73
NJN     64
NOH     40
SEA     37
NOK     18
LBN      9
CHH      7
EST      5
DRT      4
VAN      2
WST      1
Name: count, dtype: int64

- Columna position

In [16]:
df_work['position'].value_counts()

position
Guard             1358
Forward           1099
Center             441
Guard-Forward      291
Forward-Center     254
Center-Forward     156
Forward-Guard      113
Name: count, dtype: int64

- Columna g-league

In [18]:
df_work['g-league'].value_counts()

g-league
Y    1918
N    1794
Name: count, dtype: int64

- Columna nba

In [20]:
df_work['nba'].value_counts()

nba
Y    3698
N      14
Name: count, dtype: int64

- Columna greatest_75_flag

In [21]:
df_work['greatest_75_flag'].value_counts()

greatest_75_flag
N    3650
Y      62
Name: count, dtype: int64

Tras realizar el análisis de las columnas categóricas del dataset utilizando la revisión de valores únicos, no se han detectado inconsistencias en los datos. 
Por tanto, no ha sido necesario aplicar ninguna transformación adicional en esta fase de la limpieza.

***Paso 5 --> Detección de outliers***

Para detectar posibles outliers, vamos a hacer uso de la función describe sobre la totalidad del dataset de nuevo tras todas la limpieza.

In [26]:
df_work.describe()

,id,general_team,is_active,draft_team,weight,seniority,from_year,to_year,jersey_def,height_definitivo
count,3.712000e+03,3.712000e+03,3712.000000,3.712000e+03,3712.000000,3712.000000,3712.000000,3712.000000,3712.000000,3712.000000
mean,5.333399e+05,1.610613e+09,0.275323,1.230522e+09,220.919989,8.452586,2008.573545,2016.116110,19.914332,200.048265
std,6.879675e+05,2.914681e+02,0.446737,6.839854e+08,27.155020,4.950895,8.114313,6.346183,18.105151,9.073202
min,1.500000e+01,1.610613e+09,0.000000,0.000000e+00,133.000000,0.000000,1978.000000,1996.000000,0.000000,165.100000
25%,2.550000e+03,1.610613e+09,0.000000,1.610613e+09,200.000000,4.000000,2003.000000,2011.000000,6.000000,193.040000
50%,2.019540e+05,1.610613e+09,0.000000,1.610613e+09,220.000000,8.000000,2009.000000,2017.000000,15.000000,200.660000
75%,1.626204e+06,1.610613e+09,1.000000,1.610613e+09,240.000000,12.000000,2016.000000,2022.000000,30.000000,205.740000
max,1.631323e+06,1.610617e+09,1.000000,1.610613e+09,325.000000,22.000000,2022.000000,2023.000000,99.000000,231.140000


Todo parece estar en orden con respecto a la detección de outliers pero hay un valor que llama nuestra atención: 

1. Jugadores con seniority = 0

Como puede ser que haya jugadores cuya experiencia sea de 0 temporadas pero se muestren en nuestro dataset. 
Vamos a averiguarlo obteniendo todos los jugadores para los que seniority sea 0

In [29]:
print(df_work[df_work['seniority']==0][['full_name', 'seniority', 'from_year','to_year']])

             full_name  seniority  from_year  to_year
3499  Collin Gillespie        0.0     2022.0   2023.0
3512     Chet Holmgren        0.0     2022.0   2023.0
3682      Justin Lewis        0.0     2022.0   2023.0


Tras obtener los jugadores para los que seniority es igual a 0 entendemos el por que. Estos jugadores fueron drafteados en 2022 pero no llegaron a debutar en dicho año ya que se lesionaron antes del inicio de la temporada.

Por otro lado, hay valores que podrían llamar también nuestra atención como por ejemplo el valor mínimo en la columna de altura que son 165cm. 

Esto es un valor algo bajo pero se han dado casos de jugadores NBA que median menos de 165  (e.g Muggsy Bogues o Earl Boykins) por lo que  no lo consideramos un outlier.

In [ ]:
#Revisamos las alturas bajas
print(df_work[df_work['height_definitivo'] < 170][['full_name', 'weight', 'height_definitivo']])

         full_name  weight  height_definitivo
444   Earl Boykins   133.0              165.1
1581  Earl Boykins   133.0              165.1


Es el mismo caso que el valor máximo de peso en 325 lb lo que es muy alto para un jugador nba, pero se han dado casos en la historia como por ejemplo Shaquille O Neal.

In [33]:
#Revisamos los pesos altos
print(df_work[df_work['weight'] > 320][['full_name', 'weight', 'height_definitivo']])

             full_name  weight  height_definitivo
125   Shaquille O'Neal   325.0              215.9
357   Shaquille O'Neal   325.0              215.9
860   Shaquille O'Neal   325.0              215.9
1115  Shaquille O'Neal   325.0              215.9
1294  Shaquille O'Neal   325.0              215.9


Con esto, damos por concluida la limpieza y pasamos a la fase de transformación del dataset.

In [34]:
#Exportamos el dataset
df_work.to_csv("Dataframe_limpio.csv", index= False)